# Train Logistic Regression

This notebook mirrors `train_logistic_regression.py` in a notebook-friendly format. Update the config cell, then run the remaining cells from top to bottom.

In [8]:
!pip install pandas
!pip install rich
!pip install -U scikit-learn
from __future__ import annotations

import pickle
from pathlib import Path

import pandas as pd
from preprocess_preview import show_preprocessing_preview
from sklearn.callback import ProgressBar, ScoringMonitor
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler
from teaching_scalers import BadMagnitudeScaler
from training_config import TrainingConfig

In [9]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
config = TrainingConfig(
    data=Path("WA_Fn-UseC_-Telco-Customer-Churn.csv"),
    model_out=Path("logistic_regression_churn.pkl"),
    test_size=0.2,
    random_state=42,
    preview_rows=5,
    preprocessed_out=Path("preprocessed_training_data.csv"),
    categorical_encoding="onehot",
    numeric_scaler="standard",
)

config

TrainingConfig(data=PosixPath('WA_Fn-UseC_-Telco-Customer-Churn.csv'), model_out=PosixPath('logistic_regression_churn.pkl'), test_size=0.2, random_state=42, preview_rows=5, preprocessed_out=PosixPath('preprocessed_training_data.csv'), categorical_encoding='onehot', numeric_scaler='standard')

In [13]:
def load_data(csv_path: Path) -> tuple[pd.DataFrame, pd.Series]:
    data = pd.read_csv(csv_path)
    print("Raw Data:\n", data)

    # The customer id is an identifier, not a predictive signal.
    data = data.drop(columns=["customerID"])

    # Some TotalCharges values are stored as text and may contain blanks.
    data["TotalCharges"] = pd.to_numeric(data["TotalCharges"], errors="coerce")
    data["Churn"] = data["Churn"].map({"No": 0, "Yes": 1})

    print("Data Numericalize Churn:\n", data)

    features = data.drop(columns=["Churn"])
    target = data["Churn"]
    return features, target


def build_model(
    numeric_features: list[str],
    categorical_features: list[str],
    categorical_encoding: str,
    numeric_scaler: str,
) -> Pipeline:
    # Imputer defines, what value the machine should treat it by default, if missing
    # Scaler defines how to rescale the data before training.
    if numeric_scaler == "standard":
        scaler = StandardScaler()
    else:
        scaler = BadMagnitudeScaler()

    numeric_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", scaler),
        ]
    )

    if categorical_encoding == "onehot":
        encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    else:
        # this is actually a wrong way, we just demonstrate for teaching.
        encoder = OrdinalEncoder(
            handle_unknown="use_encoded_value",
            unknown_value=-1,
            encoded_missing_value=-1,
        )

    categorical_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", encoder),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_pipeline, numeric_features),
            ("cat", categorical_pipeline, categorical_features),
        ]
    )

    # f(w, X) = \sum_{i} w[i] * X[i], where w is TBD coefficients.
    # p(w, X) = exp(f(w,X)) / (exp(f(w,X)) + 1)
    #
    # loss(w) = - y ln p(w,X) - (1-y) ln (1-p(w,X))
    #
    # Training process essential: Find a w to minimize loss function
    classifier = LogisticRegression(max_iter=10, solver="lbfgs", verbose=1)

    model._categorical_encoding = categorical_encoding
    model._numeric_scaler = numeric_scaler
    return model

In [14]:
X, y = load_data(config.data)

print("X:\n", X)
print("y:\n", y)

numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

print("numeric_features:", numeric_features)
print("categorical_features:", categorical_features)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=config.test_size,
    random_state=config.random_state,
    stratify=y,
)

model = build_model(
    numeric_features,
    categorical_features,
    config.categorical_encoding,
    config.numeric_scaler,
)

print(f"Fit monitoring mode: {model._fit_monitoring_mode}")
print(f"Categorical encoding mode: {config.categorical_encoding}")
print(f"Numeric scaler mode: {config.numeric_scaler}")
model.fit(X_train, y_train)

show_preprocessing_preview(
    model=model,
    X_train=X_train,
    numeric_features=numeric_features,
    categorical_features=categorical_features,
    categorical_encoding=config.categorical_encoding,
    numeric_scaler=config.numeric_scaler,
    preview_rows=config.preview_rows,
    preprocessed_out=config.preprocessed_out,
)

classifier = model.named_steps["classifier"]
predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)

print(f"Training rows: {len(X_train)}")
print(f"Test rows: {len(X_test)}")
print(f"Solver iterations used: {classifier.n_iter_}")
print(f"Accuracy: {accuracy:.4f}")

monitor = getattr(model, "_scoring_monitor", None)
if monitor is not None:
    try:
        score_log = monitor.get_logs().data_as_pandas
        score_log = score_log.loc[
            score_log["accuracy"].notna(),
            ["task_name", "task_id", "accuracy"],
        ]
        if not score_log.empty:
            if len(score_log) > 10:
                score_log = score_log.tail(10)
            print("\nTraining accuracy snapshots:")
            print(score_log.to_string(index=False))
    except ValueError:
        pass

print("\nClassification report:")
print(classification_report(y_test, predictions, target_names=["No Churn", "Churn"]))

with config.model_out.open("wb") as model_file:
    pickle.dump(model, model_file)
print(f"Saved trained pipeline to: {config.model_out}")

Raw Data:
       customerID  gender  SeniorCitizen Partner Dependents  tenure  \
0     7590-VHVEG  Female              0     Yes         No       1   
1     5575-GNVDE    Male              0      No         No      34   
2     3668-QPYBK    Male              0      No         No       2   
3     7795-CFOCW    Male              0      No         No      45   
4     9237-HQITU  Female              0      No         No       2   
...          ...     ...            ...     ...        ...     ...   
7038  6840-RESVB    Male              0     Yes        Yes      24   
7039  2234-XADUH  Female              0     Yes        Yes      72   
7040  4801-JZAZL  Female              0     Yes        Yes      11   
7041  8361-LTMKD    Male              1     Yes         No       4   
7042  3186-AJIEK    Male              0      No         No      66   

     PhoneService     MultipleLines InternetService OnlineSecurity  ...  \
0              No  No phone service             DSL             No  ...  

TypeError: The sub-estimator (LogisticRegression) of a meta-estimator (Pipeline) can't have auto-propagated callbacks (['ProgressBar']). Register them directly on the meta-estimator.